# Information Health — one-click Colab demo

Runs the **FastAPI engine** + a **production build** of the Next.js app inside Colab and
opens a public URL.

- A *production* build is served on purpose: `next dev` injects CSS via JS and uses a
  hot-reload websocket that misbehaves behind a tunnel (blank / unstyled page). Production
  ships a real stylesheet + static chunks, so it renders reliably.
- The onboarding → **Initial Information Health Estimate** flow needs no credentials.
- **See the whole signed-in app with no Google setup:** a one-click *“Continue as demo reader”*
  login (dev-only) plus a cell that pre-loads **real catalog reads** (falling back to sample URLs on
  an empty catalog) means the **Dashboard, Reading History, Analytics, Profile — and cross-cutting
  Bridging recommendations** — are populated the moment you sign in.
- Choose the recommendation corpus in step 2: **live RSS feed** (real names + openable article URLs, so **Read** opens the publisher page), **Qbias** (real names), or **synthetic**.
- Optional: **Google sign-in**.

Run the cells top to bottom (Runtime → *Run all* also works).

> If your repo is **private**, paste a GitHub token (read access) in step 1.


In [ ]:
#@title 1 · Get the code — fresh clone, or in-place update (your data survives) + install Node 20
REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}   # only needed if the repo is private

import os
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
%cd /content
if os.path.isdir("app/.git"):
    # Existing checkout: update IN PLACE. data/ (your SQLite DB — users, reads, reports) is
    # gitignored, so the reset never touches it. Re-running this cell is always safe now.
    %cd app
    !git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD
else:
    !rm -rf app && git clone --depth 1 --branch {BRANCH} https://{auth}github.com/{REPO}.git app
    %cd app
!git log --oneline -1
# Colab ships an old Node; install a modern one for Next.js 14.
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1
!sudo apt-get install -y nodejs >/dev/null 2>&1
!node -v ; npm -v


In [ ]:
#@title 2 · Start the FastAPI engine — choose the recommendation corpus
#@markdown **live-feed** = real publishers **with openable article URLs** (ingests RSS + optional
#@markdown NewsAPI/GDELT; the Read button opens the real article). **qbias** = real names, **no** URLs.
#@markdown **synthetic** = generated names, offline. live-feed falls back to synthetic if unreachable.
CORPUS = "live-feed"  #@param ["live-feed", "qbias", "synthetic"]
#@markdown ---
#@markdown **Extra live sources** (used only when CORPUS = live-feed; RSS is always on). NewsAPI needs a
#@markdown free key from https://newsapi.org — GDELT is keyless (it may be rate-limited from Colab).
USE_NEWSAPI     = False  #@param {type:"boolean"}
NEWSAPI_API_KEY = ""     #@param {type:"string"}
USE_GDELT       = False  #@param {type:"boolean"}
#@markdown **AUTO_REFRESH** keeps the catalog live — the engine re-polls every ~10 min and hot-refreshes
#@markdown the recommendation corpus with no restart (NewsAPI free tier stays ~24h delayed regardless).
AUTO_REFRESH    = False  #@param {type:"boolean"}
#@markdown **Feature flags** — the validated beta configuration. **COACH_V2** = the intent-routed
#@markdown AI Coach (question-aware replies, follow-up chips, recommendation cards in chat; off =
#@markdown the v1 report narrator). **STORY_SLOT** = the conditional Story-Match slot (max one
#@markdown card) validated earlier.
COACH_V2   = True   #@param {type:"boolean"}
STORY_SLOT = True   #@param {type:"boolean"}
#@markdown **DEMO_ACCOUNT** serves anonymous / below-threshold visitors a stable, read-only demo
#@markdown reader (seeded in cell 5) instead of a re-picked synthetic one — its score no longer
#@markdown changes across restarts.
DEMO_ACCOUNT = True   #@param {type:"boolean"}

import subprocess, time, os, json, urllib.request
!pip install -q -e ".[serve]"

# Feature flags for the engine subprocess — set on os.environ so every corpus branch (and the
# audit/validation notebooks, which subprocess-inherit the env) sees the same configuration.
os.environ["RWE_COACH_V2"] = "1" if COACH_V2 else "0"
os.environ["RWE_STORY_SLOT"] = "1" if STORY_SLOT else "0"
os.environ["RWE_DEMO_ACCOUNT"] = ("dev:demo-exhibit@infodiet.local"
                                 if DEMO_ACCOUNT else "")

def wait(url, n=240):
    for _ in range(n):
        try:
            if urllib.request.urlopen(url, timeout=2).status == 200: return True
        except Exception: time.sleep(1)
    return False

env = {**os.environ}
if CORPUS == "live-feed":
    # Ingest cross-spectrum RSS (+ optional NewsAPI / GDELT) into the ONE FeedArticle catalog; the engine
    # then sources recommendations from it so each carries the real publisher URL (Honest URL Pass-through).
    # Best-effort: if the catalog stays under the min, the engine falls back to synthetic (no URLs).
    if USE_NEWSAPI and NEWSAPI_API_KEY:
        os.environ["RWE_NEWSAPI_ENABLED"] = "1"; os.environ["RWE_NEWSAPI_API_KEY"] = NEWSAPI_API_KEY
    if USE_GDELT:
        os.environ["RWE_GDELT_ENABLED"] = "1"
    !python examples/rss_ingest.py run --feeds deploy/rss_feeds.example.txt || true
    !python examples/sources.py poll || true      # ingest every ENABLED extra source (NewsAPI / GDELT)
    !python examples/rss_ingest.py status
    env = {**os.environ}                            # re-copy so the engine inherits the flags/key set above
    env["RWE_RECS_SOURCE"] = "feed"
    env["RWE_FEED_MAX_PER_OUTLET"] = "40"   # balance: no single feed dominates the recs
    if AUTO_REFRESH:
        env["RWE_FEED_POLL"] = "1"                  # background poller + hot refresh (the auto-refresh loop)
        env["RWE_NEWSAPI_POLL_INTERVAL"] = "1800"   # ~48 calls/day — under the free NewsAPI 100/day limit
        print("auto-refresh ON — the engine will re-poll sources and hot-refresh the recs corpus.")
elif CORPUS == "qbias":
    !wget -q "https://raw.githubusercontent.com/irgroup/Qbias/main/allsides_balanced_news_headlines-texts.csv" -O qbias.csv
    env.update({"RWE_PROFILE": "qbias", "RWE_QBIAS": "qbias.csv", "RWE_MAX_ITEMS": "1200", "RWE_N_USERS": "300"})

# IMPORTANT: kill any engine a previous cell run left alive — Colab keeps subprocesses running, so a
# stale engine would keep holding port 8000 and THIS freshly-configured one would never take over.
subprocess.run(["pkill", "-f", "examples/api_fastapi.py"], stderr=subprocess.DEVNULL); time.sleep(2)
engine = subprocess.Popen(["python", "examples/api_fastapi.py"], env=env,
                          stdout=open("engine.log", "w"), stderr=subprocess.STDOUT)
up = wait("http://127.0.0.1:8000/api/health")
print("engine:", "UP" if up else "FAILED — see engine.log")
print("coach:", "v2 — intent-routed (chips + cards; needs a signed-in reader with 5+ reads)"
      if COACH_V2 else "v1 — report narrator")

# Unmissable diagnostic: is the live feed actually driving recommendations (real, openable URLs)?
if up:
    health = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/api/health").read())
    src = health.get("recommendationSource")
    if src is None:
        print("WARNING: the engine on :8000 predates the recs-source diagnostic — a stale engine is")
        print("  likely still bound to the port. Do: Runtime -> Restart session, then Run all.")
    else:
        recs = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/api/recommendations", timeout=20).read())
        withurl = [r for r in recs if r["article"].get("url")]
        print(f"recommendation source: {src['source']}  (feed articles in catalog: {src['feedArticles']})")
        print(f"recommendations: {len(recs)} — with a real, openable publisher URL: {len(withurl)}")
        if withurl:
            print("  e.g.", withurl[0]["article"]["publisher"], "->", withurl[0]["article"]["url"])
            print("  -> Recommendations: each card shows \"Read article\" and opens the real publisher page.")
        elif CORPUS == "live-feed":
            print("  WARNING: feeds unreachable or <50 articles -> fell back to synthetic (no URLs).")
            print("  Re-run this cell, or edit deploy/rss_feeds.example.txt; check: !python examples/rss_ingest.py status")

In [ ]:
#@title 3 · Build the web app (production) + start it, pointed at the engine
import secrets
# Production is served (dev mode's HMR + JS-injected CSS break behind a tunnel).
# Prod also disables the mock fallback, so it uses the real engine from step 2.
# RWE_DEV_LOGIN / NEXT_PUBLIC_DEV_LOGIN turn on a one-click "Continue as demo reader" sign-in so you
# can explore the full signed-in app (Dashboard, History, Analytics, Profile, Settings) without
# Google OAuth. Dev/demo ONLY — never set these in a real deployment.
open("web/.env.local", "w").write(
    "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
    f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
    "RWE_DEV_LOGIN=1\n"
    "NEXT_PUBLIC_DEV_LOGIN=1\n")
!cd web && npm install --no-audit --no-fund --loglevel=error
!cd web && npm run build
# Kill any web server a previous run left alive (same reason as the engine in cell 2 — a stale
# next-server would keep holding :3000 and this freshly-built bundle would never take over).
subprocess.run(["pkill", "-f", "next-server"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "next start"], stderr=subprocess.DEVNULL); time.sleep(2)
web = subprocess.Popen(["npm", "start"], cwd="web",
                       stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
print("web:", "UP" if wait("http://127.0.0.1:3000/onboarding") else "still starting — check web.log")

In [ ]:
#@title 4 · Open it — public URL via a Cloudflare quick tunnel  (safe to re-run)
import re, time, subprocess, secrets, os

# Clean any previous tunnel so re-running this cell is idempotent.
subprocess.run(["pkill", "-f", "cloudflared"], stderr=subprocess.DEVNULL); time.sleep(1)
# (Re)download cloudflared if it's missing or truncated.
if not os.path.exists("cloudflared") or os.path.getsize("cloudflared") < 1_000_000:
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   "cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)
print("cloudflared binary:", os.path.getsize("cloudflared"), "bytes")

open("cf.log", "w").close()
cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
                      stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(90):                       # quick tunnels can take 20-40s — wait up to ~90s
    time.sleep(1)
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", open("cf.log").read())
    if m: PUBLIC_URL = m.group(0); break

if not PUBLIC_URL:
    print("\n⚠️  No tunnel URL yet — Cloudflare quick tunnels are occasionally slow/rate-limited.")
    print("    Just RE-RUN THIS CELL (it usually works on the 2nd try). Recent cloudflared log:")
    print("    " + "\n    ".join(open("cf.log").read().splitlines()[-12:] or ["(empty)"]))
else:
    # Point NextAuth at the public tunnel URL and restart the web server, so sign-in redirects target
    # the tunnel (not the container's localhost). Runtime env only — no rebuild, so the demo-login
    # button baked in step 3 stays; the dev demo login stays enabled.
    open("web/.env.local", "w").write(
        "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
        f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
        f"NEXTAUTH_URL={PUBLIC_URL}\n"
        "RWE_DEV_LOGIN=1\n"
        "NEXT_PUBLIC_DEV_LOGIN=1\n")
    subprocess.run(["pkill", "-f", "next-server"], stderr=subprocess.DEVNULL); time.sleep(2)
    web = subprocess.Popen(["npm", "start"], cwd="web",
                           stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
    wait("http://127.0.0.1:3000/onboarding")
    print("\n👉  Open:", PUBLIC_URL)
    print("    Anonymous : /onboarding → pick publishers → Initial Estimate → Report / Recommendations / Coach")
    print("    Whole app : Sign in → “Continue as demo reader” (the only button) → Dashboard / History / Analytics / Profile")
    print("    (run step 5 first to pre-load the demo reader with sample reads)")

In [ ]:
#@title 5 · Pre-load the demo reader with real reads (so the signed-in pages have data)
# Creates the SAME throwaway demo account the "Continue as demo reader" button signs into and records
# enough reads to cross the measured-report threshold, so Dashboard / History / Analytics / Report
# populate the moment you sign in. Dev/demo only (no RWE_INTERNAL_SECRET set in this demo).
#
# The reads are REAL catalog articles (from /api/discover): a read of a catalog article lands on that
# article's real column in the recommendation corpus, keeping the demo reader CONNECTED to the click
# graph — so RWE-B cross-cutting/bridging (and the Political-openness slider) work out of the box.
# Composition: mostly left + some centre/right — a left-leaning diet, so bridges point right.
# Fallback: if the catalog is empty (CORPUS=synthetic, or feeds unreachable) the old illustrative
# sample URLs are seeded instead — every page still populates, but bridging stays inactive until the
# reader opens real articles (fabricated URLs have no column in the corpus to connect to).
import json, urllib.error, urllib.request
ENGINE = "http://127.0.0.1:8000"
def _req(method, path, body=None, headers=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(ENGINE + path, data=data, method=method,
                                 headers={"Content-Type": "application/json", **(headers or {})})
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())

uid = _req("POST", "/api/internal/users",
           {"provider": "dev", "providerAccountId": "demo@infodiet.local",
            "email": "demo@infodiet.local", "displayName": "Demo Reader"})["userId"]

try:
    arts = _req("GET", "/api/discover?limit=200").get("articles") or []
except Exception:
    arts = []

# 4 left + 2 centre + 2 right from the live catalog = a varied, left-leaning diet (bridges point right).
seen, picks = set(), []
def _take(pred, n):
    got = 0
    for a in arts:
        if got >= n:
            break
        u = a.get("url")
        if u and u not in seen and pred(float(a.get("lean") or 0.0)):
            seen.add(u); picks.append(a); got += 1
_take(lambda l: l < -0.5, 4)
_take(lambda l: -0.5 <= l <= 0.5, 2)
_take(lambda l: l > 0.5, 2)
_take(lambda l: True, 8 - len(picks))          # top up from anything unread if a bucket ran dry

if len(picks) >= 5:
    reads = [{"url": a["url"], "title": a.get("headline", "")} for a in picks]
    src = f"{len(picks)} REAL catalog articles (connected — bridging-ready)"
else:
    SAMPLE = [
        ("https://www.nytimes.com/2026/us/politics/senate-vote", "Senate advances the funding bill, leaders say"),
        ("https://www.foxnews.com/politics/border-plan", "Outrage as officials slam the border plan"),
        ("https://www.wsj.com/economy/inflation-opinion", "Opinion: why we must rethink inflation policy"),
        ("https://www.washingtonpost.com/politics/court-analysis", "Analysis: what to know about the court ruling"),
        ("https://www.theguardian.com/us-news/climate-deal", "Hope as historic climate deal is celebrated"),
        ("https://apnews.com/hub/politics/poll", "Poll finds shifting views on the economy, new data shows"),
        ("https://www.npr.org/2026/health/study", "Study finds a new treatment shows promise, researchers say"),
        ("https://www.cnn.com/2026/tech/ai-regulation", "Lawmakers clash over AI regulation amid fierce debate"),
    ]
    reads = [{"url": u, "title": t} for u, t in SAMPLE]
    src = "illustrative sample URLs (catalog empty — bridging inactive until real articles are read)"

res = _req("POST", "/api/me/reads", {"reads": reads}, {"X-IH-User-Id": str(uid)})
print(f"demo reader #{uid}: seeded {src}")
print(f"  accepted {res['accepted']} (duplicates {res['duplicates']}) — total reads "
      f"{res['totalReads']}; measured report ready: {res['sufficient']}")
print("→ open the URL, Sign in → “Continue as demo reader”: Dashboard, History, Analytics and")
print("  Recommendations are populated" + (" — including cross-cutting Bridging cards."
      if len(picks) >= 5 else " (Bridging activates once you read real catalog articles)."))

# ---- The read-only EXHIBIT account (RWE_DEMO_ACCOUNT, cell 2) ----
# What ANONYMOUS / below-threshold visitors see. Seeded once through the same public pipeline;
# the account LOCKS itself the moment it crosses the read threshold (administrative writes 403
# from then on), so its report is stable across restarts — no more wandering demo score.
from datetime import datetime, timedelta, timezone
ex_uid = _req("POST", "/api/internal/users",
              {"provider": "dev", "providerAccountId": "demo-exhibit@infodiet.local",
               "email": "demo-exhibit@infodiet.local", "displayName": "Demo Reader"})["userId"]
picks.clear()                       # fresh articles for the exhibit (seen excludes the login's)
_take(lambda l: l < -0.5, 4); _take(lambda l: -0.5 <= l <= 0.5, 2); _take(lambda l: l > 0.5, 2)
_take(lambda l: True, 8 - len(picks))
if len(picks) >= 5:
    ex_reads = [{"url": a["url"], "title": a.get("headline", ""),
                 "observedAt": (datetime.now(timezone.utc) - timedelta(days=1 + i)).isoformat()}
                for i, a in enumerate(picks)]
else:
    ex_reads = [{"url": u + "?exhibit=1", "title": t} for u, t in SAMPLE]
try:
    ex = _req("POST", "/api/me/reads", {"reads": ex_reads}, {"X-IH-User-Id": str(ex_uid)})
    print(f"exhibit account #{ex_uid}: seeded {ex['totalReads']} reads — "
          f"locked read-only: {ex['sufficient']} (anonymous visitors now see THIS reader)")
except urllib.error.HTTPError as e:
    if e.code != 403:
        raise
    # already seeded on a previous run — the one-way lock is doing its job; re-running is safe
    print(f"exhibit account #{ex_uid}: already seeded & locked read-only ✓ (re-run is a no-op)")


In [ ]:
#@title (optional) 6 · Configure Google OAuth, then restart the web server
CLIENT_ID     = ""  #@param {type:"string"}
CLIENT_SECRET = ""  #@param {type:"string"}
import secrets, time, subprocess
assert PUBLIC_URL, "run step 4 first to get the public URL"
open("web/.env.local", "w").write(
    "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
    f"GOOGLE_CLIENT_ID={CLIENT_ID}\n"
    f"GOOGLE_CLIENT_SECRET={CLIENT_SECRET}\n"
    f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
    f"NEXTAUTH_URL={PUBLIC_URL}\n"
    "RWE_DEV_LOGIN=1\n")   # keep the one-click demo login available alongside Google
# server-side env is read when the server starts, so a restart is enough (no rebuild).
web.terminate(); time.sleep(2)
web = subprocess.Popen(["npm", "start"], cwd="web",
                       stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
print("Add this redirect URI to your Google OAuth client, then reload the app:")
print(" ", PUBLIC_URL + "/api/auth/callback/google")
print("web restarting:", "UP" if wait("http://127.0.0.1:3000/onboarding") else "check web.log")